# Walk-forward predictor: does an 'engine' LEARN to predict next week's close? (Colab, token-free)

Strict, no-look-ahead test (Jake's spec): daily OHLCV from 2020; **2020 = learning only**; from **Jan 2021**
go ONE WEEK at a time, retrain on everything up to that week, predict the FOLLOWING week's close. Track (a)
DIRECTION hit-rate and (b) whether the % error SHRINKS over time (= learning).

## The two honesty rails (why most versions of this fool people)
1. **Direction bar = the up-week base rate (~56%), NOT 50%.** 'Always predict up' already scores the base
   rate. The engine must beat THAT, not a coin.
2. **Closeness bar = PERSISTENCE** ('next close = this close'), measured in **% error** (not points — the
   index level rises, which fakes 'getting closer').
Retrains weekly on an expanding window, so it genuinely CAN learn. Run top-to-bottom; last cell exports.


## capture setup


In [ ]:
import builtins as _bi
REPORT_LINES=[]; _realprint=_bi.print
def print(*a, sep=' ', end='\n', **k):
    _realprint(*a, sep=sep, end=end, **k)
    try: REPORT_LINES.append(sep.join(str(x) for x in a))
    except Exception: pass
_realprint('capture ON')


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, numpy as np
    from sklearn.ensemble import RandomForestRegressor
except Exception:
    _pip('pandas'); _pip('numpy'); _pip('scikit-learn')
    import pandas as pd, numpy as np
    from sklearn.ensemble import RandomForestRegressor

TICKER = 'SPY'   # configurable: 'SPY', '^GSPC', 'QQQ', 'MU', ...

def load_daily(sym):
    try:
        import yfinance as yf
    except Exception:
        _pip('yfinance'); import yfinance as yf
    df = yf.download(sym, start='2019-06-01', end=None, progress=False, auto_adjust=True)
    if not len(df): raise RuntimeError('no data for '+sym)
    df = df.rename(columns=str.title)
    keep = {c: (c if isinstance(c,str) else c[0]) for c in df.columns}
    df.columns = [c if isinstance(c,str) else c[0] for c in df.columns]
    return df[['Open','High','Low','Close','Volume']].dropna()

daily = load_daily(TICKER)
print(TICKER, 'daily bars', len(daily), daily.index[0].date(), '->', daily.index[-1].date())


## Resample to weekly (Fri) + build PAST-ONLY features and the next-week target


In [ ]:
wk = pd.DataFrame({
    'close':  daily['Close'].resample('W-FRI').last(),
    'high':   daily['High'].resample('W-FRI').max(),
    'low':    daily['Low'].resample('W-FRI').min(),
    'vol':    daily['Volume'].resample('W-FRI').sum(),
}).dropna()
wk['r1']  = wk['close'].pct_change()
wk['r4']  = wk['close'].pct_change(4)
wk['r12'] = wk['close'].pct_change(12)
wk['vol4']= wk['r1'].rolling(4).std()
wk['rng'] = (wk['high']-wk['low'])/wk['close']
wk['madist'] = wk['close']/wk['close'].rolling(10).mean() - 1
wk['dvol'] = wk['vol'].pct_change()
FEATS = ['r1','r4','r12','vol4','rng','madist','dvol']
wk['target'] = wk['close'].shift(-1)/wk['close'] - 1     # next-week return (known one week later)
wk = wk.dropna(subset=FEATS)
print('weekly rows with features:', len(wk), '| first', wk.index[0].date(), 'last', wk.index[-1].date())


## Walk-forward: retrain each week on ALL prior weeks, predict the next week's close


In [ ]:
START = '2021-01-01'   # predictions begin here; everything before (2020) is training-only
rows = []
idx = wk.index
# a week t is predictable if we have features at t AND the outcome (t->t+1) will be observed
pred_weeks = [i for i in range(len(wk)-1) if idx[i] >= pd.Timestamp(START)]
for i in pred_weeks:
    # TRAIN on all j < i whose target is realized (j from 0..i-1). No look-ahead: uses only weeks up to t.
    tr = wk.iloc[:i].dropna(subset=['target'])
    if len(tr) < 40: continue
    m = RandomForestRegressor(n_estimators=200, max_depth=3, min_samples_leaf=5, random_state=42, n_jobs=-1)
    m.fit(tr[FEATS].values, tr['target'].values)
    x = wk[FEATS].iloc[i:i+1].values
    pred_ret = float(m.predict(x)[0])
    this_close = wk['close'].iloc[i]
    actual_close = wk['close'].iloc[i+1]
    actual_ret = actual_close/this_close - 1
    rows.append(dict(date=idx[i+1], this_close=this_close, pred_close=this_close*(1+pred_ret),
                     actual_close=actual_close, pred_ret=pred_ret, actual_ret=actual_ret))
R = pd.DataFrame(rows).set_index('date')
print('predictions made:', len(R), '| from', R.index[0].date(), 'to', R.index[-1].date())


## Results — vs the two honest baselines, and did it LEARN?


In [ ]:
R['pred_up']   = R['pred_ret']   > 0
R['actual_up'] = R['actual_ret'] > 0
R['dir_hit']   = R['pred_up'] == R['actual_up']
R['ape']       = (R['pred_close']-R['actual_close']).abs()/R['actual_close']        # model % error
R['ape_persist']=(R['this_close']-R['actual_close']).abs()/R['actual_close']        # persistence % error

base_up = R['actual_up'].mean()          # 'always predict up' score = up-week base rate
print('='*60); print('  DIRECTION'); print('='*60)
print(f'  engine direction hit-rate : {100*R["dir_hit"].mean():.1f}%')
print(f'  base rate (always-up)     : {100*base_up:.1f}%   <-- the bar to beat')
print(f'  edge over always-up       : {100*(R["dir_hit"].mean()-max(base_up,1-base_up)):+.1f} pts',
      '  (best dumb rule = always the majority side)')
print('\n'+'='*60); print('  CLOSENESS (% error, lower better)'); print('='*60)
print(f'  engine MAPE      : {100*R["ape"].mean():.2f}%')
print(f'  persistence MAPE : {100*R["ape_persist"].mean():.2f}%   <-- the bar to beat')
print(f'  engine beats persistence? {"YES" if R["ape"].mean()<R["ape_persist"].mean() else "NO"}')

print('\n'+'='*60); print('  DID IT LEARN? (first half vs second half)'); print('='*60)
h = len(R)//2; H1, H2 = R.iloc[:h], R.iloc[h:]
print(f'  1st half: dir {100*H1["dir_hit"].mean():.1f}% | MAPE {100*H1["ape"].mean():.2f}%  ({H1.index[0].date()}..{H1.index[-1].date()})')
print(f'  2nd half: dir {100*H2["dir_hit"].mean():.1f}% | MAPE {100*H2["ape"].mean():.2f}%  ({H2.index[0].date()}..{H2.index[-1].date()})')
print('  LEARNING = 2nd-half direction UP and/or MAPE DOWN vs 1st half. Flat = it did NOT learn.')
print('\n  rolling 26wk direction hit-rate (watch for an uptrend = learning):')
roll = R['dir_hit'].rolling(26).mean().dropna()
for dt,v in list(roll.items())[::13]: print(f'    {dt.date()}: {100*v:.0f}%')


## ⬇️ EXPORT — run LAST


In [ ]:
from datetime import datetime
fname='walkforward_report.txt'
hdr=['='*60,f'WALK-FORWARD PREDICTOR ({TICKER}) — export',
     f'predictions {R.index[0].date()}..{R.index[-1].date()} | n={len(R)}',
     f'generated {datetime.now().strftime("%Y-%m-%d %H:%M")}','='*60,'']
open(fname,'w').write('\n'.join(hdr)+'\n'+'\n'.join(REPORT_LINES)+'\n')
_realprint('wrote',fname,'—',len(REPORT_LINES),'lines')
if not REPORT_LINES: _realprint('!! empty — run cells above first')
try:
    from google.colab import files; files.download(fname); _realprint('download started')
except Exception as e:
    _realprint('not in Colab / blocked:',e,'- grab it from the folder icon (left)')


### How to read
- **Direction:** the number that matters is *edge over always-up*. ~0 (or negative) = the engine learned
  nothing that beats a dumb majority rule. A few points POSITIVE and stable = a real (if tiny) directional edge.
- **Closeness:** if engine MAPE ≥ persistence MAPE, predicting price added nothing over 'next = this week'.
- **Learning:** flat first-half vs second-half (and a flat rolling line) = it did NOT learn — more data didn't
  help, because the signal isn't there. That's the honest, likely answer — and it's the point of the test.
- Change `TICKER` to try another name. Paste the report back and we'll read the verdict.
